In [14]:
# Importar bibliotecas

import pandas as pd
import numpy as np

In [15]:
# Carregar a base tratada

df = pd.read_csv("../data/processed/customer_churn_clean.csv")

In [16]:
df.shape

(7043, 21)

In [17]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [18]:
# Criar variável alvo para o modelo

df_model = df.copy()

df_model["ChurnFlag"] = (
    df_model["Churn"]
    .map({
        "No": 0,
        "Yes": 1
    })
    .astype(int)
)

In [19]:
df_model["ChurnFlag"].value_counts()

ChurnFlag
0    5174
1    1869
Name: count, dtype: int64

In [20]:
df_model["ChurnFlag"].isna().sum()

np.int64(0)

In [21]:
# Identificar as variáveis numéricas e categóricas

variaveis_numericas = df_model.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

variaveis_categoricas = df_model.select_dtypes(
    include=["object"]
).columns.tolist()

print("Variáveis numéricas:")
print(variaveis_numericas)

print("\nVariáveis categóricas:")
print(variaveis_categoricas)

Variáveis numéricas:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'ChurnFlag']

Variáveis categóricas:
['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn']


/var/folders/zh/rfmp_tpx19185x13n9pr4q4w0000gn/T/ipykernel_3812/1670539804.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  variaveis_categoricas = df_model.select_dtypes(


In [22]:
# Separar as variáveis preditoras da variável alvo

X = df_model.drop(
    columns=["customerID", "Churn", "ChurnFlag"]
)

y = df_model["ChurnFlag"]

print("Dimensão de X:", X.shape)
print("Dimensão de y:", y.shape)

Dimensão de X: (7043, 19)
Dimensão de y: (7043,)


In [23]:
# Verificar as variáveis preditoras que serão utilizadas no modelo

print(X.columns.tolist())

['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']


In [24]:
# Instalar a biblioteca necessária para a preparação do modelo

%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [25]:
# Preparar as variáveis categóricas e numéricas para o modelo

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

variaveis_numericas = X.select_dtypes(
    include=["int64", "float64"]
).columns

variaveis_categoricas = X.select_dtypes(
    include=["object"]
).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", variaveis_numericas),
        ("cat", OneHotEncoder(handle_unknown="ignore"), variaveis_categoricas)
    ]
)

/var/folders/zh/rfmp_tpx19185x13n9pr4q4w0000gn/T/ipykernel_3812/4063335164.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  variaveis_categoricas = X.select_dtypes(


In [26]:
# Verificar a quantidade de variáveis após a transformação

X_transformado = preprocessor.fit_transform(X)

print("Dimensão antes da transformação:", X.shape)
print("Dimensão após a transformação:", X_transformado.shape)

Dimensão antes da transformação: (7043, 19)
Dimensão após a transformação: (7043, 45)


In [27]:
# Registrar a dimensão final dos dados preparados para modelagem

print("Dados preparados para modelagem:", X_transformado.shape)
print("Variável alvo:", y.shape)

Dados preparados para modelagem: (7043, 45)
Variável alvo: (7043,)
